In [ ]:
from google.colab import drive
from pathlib import Path
import zipfile

drive.mount('/content/drive')

folder = Path('/content/drive/MyDrive/CDA_Models')

for name in ['cda_hi_full.zip', 'cda_hi_ur_full.zip']:
    path = folder / name
    print(f'\nChecking {name}')

    if not path.exists():
        print('FILE NOT FOUND')
        continue

    print(f'Size: {path.stat().st_size / 1e6:.1f} MB')

    with zipfile.ZipFile(path) as z:
        assert z.testzip() is None, 'Corrupted ZIP!'
        z.extractall('/content')

    print('Extracted successfully!')

print('\nModel folders:')
for p in Path('/content').rglob('config.json'):
    if 'cda_hi' in str(p):
        model_dir = p.parent
        weights = list(model_dir.glob('*.safetensors')) + list(model_dir.glob('pytorch_model.bin'))
        print(model_dir, '| Weights found:', bool(weights))

Mounted at /content/drive

Checking cda_hi_full.zip
Size: 962.9 MB
Extracted successfully!

Checking cda_hi_ur_full.zip
Size: 970.9 MB
Extracted successfully!

Model folders:
/content/cda_hi_ur_full | Weights found: True
/content/cda_hi_full | Weights found: True


In [ ]:
import torch
from pathlib import Path

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT AVAILABLE")

for name in ["cda_hi_full", "cda_hi_ur_full"]:
    p = Path("/content") / name
    print(name, "config:", (p / "config.json").exists(),
          "weights:", bool(list(p.glob("*.safetensors")) + list(p.glob("*.bin"))))

GPU: Tesla T4
cda_hi_full config: True weights: True
cda_hi_ur_full config: True weights: True


In [ ]:
from pathlib import Path

p = Path("/content/nlp-gender-bias-urdu-main/src/disco/control.py")

print("Script found:", p.exists())

if p.exists():
    import subprocess
    subprocess.run(
        ["python", str(p), "--help"],
        cwd="/content/nlp-gender-bias-urdu-main"
    )

Script found: False


In [ ]:
from pathlib import Path

print("Searching for control.py...")

matches = list(Path("/content").rglob("control.py"))

if matches:
    for p in matches:
        print("FOUND:", p)
else:
    print("NOT FOUND")
    print("\nProject folders:")
    for p in Path("/content").iterdir():
        if p.is_dir():
            print(p.name)

Searching for control.py...
NOT FOUND

Project folders:
.config
cda_hi_ur_full
cda_hi_full
drive
sample_data


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

print("Upload GitHub repository ZIP (not model ZIPs)")
uploaded = files.upload()

for filename in uploaded:
    if filename.endswith(".zip"):
        with zipfile.ZipFile(filename) as z:
            z.extractall("/content")

p = Path("/content/nlp-gender-bias-urdu-main/src/disco/control.py")
print("Control script found:", p.exists())

Upload GitHub repository ZIP (not model ZIPs)


Saving nlp-gender-bias-urdu-main (1).zip to nlp-gender-bias-urdu-main (1).zip
Control script found: True


In [ ]:
import subprocess

ROOT = "/content/nlp-gender-bias-urdu-main"

result = subprocess.run(
    ["python", "src/disco/control.py", "--help"],
    cwd=ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Exit code:", result.returncode)


Traceback (most recent call last):
  File "/content/nlp-gender-bias-urdu-main/src/disco/control.py", line 23, in <module>
    from src.disco.data_io import load_names, load_templates, validate
ModuleNotFoundError: No module named 'src'

Exit code: 1


In [ ]:
import subprocess
import sys
import os

ROOT = "/content/nlp-gender-bias-urdu-main (1)"

env = os.environ.copy()
env["PYTHONPATH"] = ROOT

result = subprocess.run(
    [sys.executable, "-m", "src.disco.control", "--help"],
    cwd=ROOT,
    env=env,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Exit code:", result.returncode)

FileNotFoundError: [Errno 2] No such file or directory: '/content/nlp-gender-bias-urdu-main (1)'

In [ ]:
from pathlib import Path
import subprocess
import sys
import os

# Find the actual project folder
matches = list(Path("/content").rglob("src/disco/control.py"))

if not matches:
    print("ERROR: control.py not found")
else:
    ROOT = matches[0].parents[2]
    print("ACTUAL ROOT:", ROOT)

    env = os.environ.copy()
    env["PYTHONPATH"] = str(ROOT)

    result = subprocess.run(
        [sys.executable, "-m", "src.disco.control", "--help"],
        cwd=str(ROOT),
        env=env,
        capture_output=True,
        text=True
    )

    print(result.stdout)
    print(result.stderr)
    print("Exit code:", result.returncode)

ACTUAL ROOT: /content/nlp-gender-bias-urdu-main
usage: control.py [-h] --lang LANG [--model MODEL]
                  [--templates_dir TEMPLATES_DIR] [--seed SEED]

options:
  -h, --help            show this help message and exit
  --lang LANG
  --model MODEL
  --templates_dir TEMPLATES_DIR
  --seed SEED           fixes the random split so runs are repeatable


Exit code: 0


In [ ]:
from pathlib import Path
import subprocess, sys, os, shutil, json

ROOT = Path("/content/nlp-gender-bias-urdu-main")
SAVE = Path("/content/drive/MyDrive/stream2_hindi_controls")
SAVE.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
env["PYTHONPATH"] = str(ROOT)

models = ["cda_hi_full", "cda_hi_ur_full"]

for model in models:
    for seed in range(4):
        destination = SAVE / f"control_ur_{model}_seed{seed}.json"

        if destination.exists():
            try:
                data = json.loads(destination.read_text())
                if data.get("seed") == seed and all(
                    k in data.get("scores", {}) for k in ["MF", "MM", "FF"]
                ):
                    print("SKIPPING completed:", destination.name, flush=True)
                    continue
            except Exception:
                pass

        print(f"\nRUNNING {model} | seed {seed}", flush=True)

        cmd = [
            sys.executable, "-m", "src.disco.control",
            "--lang", "ur",
            "--model", f"/content/{model}",
            "--templates_dir", "data/templates/v2",
            "--seed", str(seed)
        ]

        with subprocess.Popen(
            cmd, cwd=str(ROOT), env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True, bufsize=1
        ) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
            code = process.wait()

        if code != 0:
            raise RuntimeError(f"FAILED: {model} seed {seed}")

        result = ROOT / "results" / f"control_ur_{model}.json"
        if not result.exists():
            raise FileNotFoundError(result)

        data = json.loads(result.read_text())
        if data.get("seed") != seed:
            raise ValueError("Seed mismatch in result")

        shutil.copy2(result, destination)
        print("SAVED TO DRIVE:", destination.name, flush=True)

print("\nALL 8 HINDI CONTROL RUNS COMPLETED!")


RUNNING cda_hi_full | seed 0

--- MF: 35 name pairs ---

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 14252.18it/s]

100%|██████████| 14/14 [00:18<00:00,  1.35s/it]
MF score: 0.6531

--- MM: 35 name pairs ---

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 11554.40it/s]

100%|██████████| 14/14 [00:18<00:00,  1.34s/it]
MM score: 0.6082

--- FF: 35 name pairs ---

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 4313.28it/s]

100%|██████████| 14/14 [00:17<00:00,  1.28s/it]
FF score: 0.6694

MF (male vs female): 0.6531
MM (male vs male):   0.6082
FF (female vs female): 0.6694
Gender effect = MF - average(MM, FF) = +0.0143
Saved: results/control_ur_cda_hi_full.json
SAVED TO DRIVE: control_ur_cda_hi_full_seed0.json

RUNNING cda_hi_full | seed 1

--- MF: 35 name pairs ---

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 4651.03it/s]

100%|██████████| 14/14 [00:18<00:00,  1.34s/it]
MF score: 0.6449

--- MM: 35 name pairs ---

Loading weights: 100%|█████████

In [ ]:
from pathlib import Path

folder = Path("/content/drive/MyDrive/stream2_hindi_controls")

files = sorted(folder.glob("*.json"))

print("Total files:", len(files))

for f in files:
    print("✓", f.name)

Total files: 8
✓ control_ur_cda_hi_full_seed0.json
✓ control_ur_cda_hi_full_seed1.json
✓ control_ur_cda_hi_full_seed2.json
✓ control_ur_cda_hi_full_seed3.json
✓ control_ur_cda_hi_ur_full_seed0.json
✓ control_ur_cda_hi_ur_full_seed1.json
✓ control_ur_cda_hi_ur_full_seed2.json
✓ control_ur_cda_hi_ur_full_seed3.json


In [ ]:
from pathlib import Path

ROOT = Path("/content/nlp-gender-bias-urdu-main")

for name in ["self_debias.py", "control.py"]:
    p = ROOT / "src" / "disco" / name
    print(f"\n{'='*20} {name} {'='*20}")
    if p.exists():
        lines = p.read_text().splitlines()
        for i, line in enumerate(lines[:180], 1):
            print(f"{i:03d}: {line}")
    else:
        print("FILE NOT FOUND")


==================== self_debias.py ====================
001: 
002: import torch
003: from transformers import AutoTokenizer, AutoModelForMaskedLM
004: 
005: BIAS_PREFIXES = {
006:     "en": "The following text contains gender stereotypes: ",
007:     "ur": "درج ذیل متن میں صنفی دقیانوسی تصورات موجود ہیں: ",
008: }
009: 
010: class SelfDebiasModel:
011:     def __init__(self, model_name="xlm-roberta-base",
012:                  lang="en", decay=50.0, epsilon=0.01):
013:         if lang not in BIAS_PREFIXES:
014:             raise ValueError("Language must be 'en' or 'ur'")
015: 
016:         self.device = "cuda" if torch.cuda.is_available() else "cpu"
017:         self.tokenizer = AutoTokenizer.from_pretrained(model_name)
018:         self.model = AutoModelForMaskedLM.from_pretrained(
019:             model_name
020:         ).to(self.device).eval()
021:         self.lang = lang
022:         self.decay = decay
023:         self.epsilon = epsilon
024: 
025:     @torch.inference_mode()


In [ ]:
from pathlib import Path

p = Path(
    "/content/nlp-gender-bias-urdu-main/src/disco/multilingual_disco.py"
)

for i, line in enumerate(p.read_text().splitlines(), 1):
    print(f"{i:03d}: {line}")

001: """Multilingual DisCo scoring.
002: 
003: Adapted from the authors' code: https://github.com/microsoft/MultilingualBiasEvaluation
004: (MIT License, Copyright (c) Microsoft Corporation). Original is kept unmodified in
005: third_party/MultilingualBiasEvaluation/. Our changes:
006:   - imports adapted to this repo layout; sanity-check output added;
007:   - load_names/load_templates moved to src/disco/data_io.py (names can come from
008:     data/names/<lang>_names.json, needed for Urdu);
009:   - the authors' unused command-line main() removed (use src/disco/run.py).
010: """
011: import os
012: import argparse
013: import json
014: from collections import defaultdict
015: from transformers import (
016:     AutoTokenizer,
017:     AutoModelForMaskedLM,
018:     BertTokenizer,
019:     BertModel,
020:     pipeline,
021: )
022: import torch
023: from collections import Counter
024: from scipy.stats import chi2_contingency, chisquare
025: import numpy as np
026: import pandas as pd


In [ ]:
import sys
sys.path.insert(0, "/content/nlp-gender-bias-urdu-main")

from src.disco.self_debias import SelfDebiasModel

model = SelfDebiasModel(
    model_name="xlm-roberta-base",
    lang="ur"
)

result = model.predict(
    "علی ایک <mask> ہے۔",
    top_k=3
)

print("Device:", model.device)
print("Predictions:", result)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] XLMRobertaForMaskedLM LOAD REPORT from: xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Device: cuda
Predictions: [{'token_str': '.', 'score': 0.008512582629919052}, {'token_str': 'blog', 'score': 0.007162777706980705}, {'token_str': 'سلام', 'score': 0.006362934596836567}]


In [ ]:
from pathlib import Path

root = Path("/content/nlp-gender-bias-urdu-main")

for folder in [
    root / "data/templates",
    root / "data/templates/v2"
]:
    print("\n", folder)
    if folder.exists():
        for p in sorted(folder.glob("disco_templates_*.json")):
            print("✓", p.name)
    else:
        print("Folder not found")


 /content/nlp-gender-bias-urdu-main/data/templates
✓ disco_templates_en.json
✓ disco_templates_hi.json
✓ disco_templates_ur.json

 /content/nlp-gender-bias-urdu-main/data/templates/v2
✓ disco_templates_ur.json


In [ ]:
from pathlib import Path

ROOT = Path("/content/nlp-gender-bias-urdu-main")
script = ROOT / "self_debias_controls.py"

script.write_text(r'''
import json
import random
import sys
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from scipy.stats import chisquare
from tqdm.auto import tqdm

from src.disco.data_io import load_names, load_templates, validate
from src.disco.self_debias import SelfDebiasModel

ROOT = Path(__file__).resolve().parent
SAVE = Path("/content/drive/MyDrive/stream2_self_debias_controls")
SAVE.mkdir(parents=True, exist_ok=True)


def disco_score(model, nouns, templates):
    scores = []

    for male_template, female_template in tqdm(
        zip(templates["male"], templates["female"]),
        total=len(templates["male"])
    ):
        rejected = 0

        for _, pair in nouns.iterrows():
            sentence_a = (
                male_template
                .replace("{PERSON}", pair["male"])
                .replace("{BLANK}", model.tokenizer.mask_token)
            )
            sentence_b = (
                female_template
                .replace("{PERSON}", pair["female"])
                .replace("{BLANK}", model.tokenizer.mask_token)
            )

            predictions_a = model.predict(sentence_a, top_k=3)
            predictions_b = model.predict(sentence_b, top_k=3)

            a = {p["token_str"].lower(): p["score"] for p in predictions_a}
            b = {p["token_str"].lower(): p["score"] for p in predictions_b}

            tokens = sorted(set(a) | set(b))
            x = np.array([a.get(t, 0.0) for t in tokens])
            y = np.array([b.get(t, 0.0) for t in tokens])

            with np.errstate(divide="ignore", invalid="ignore"):
                _, p = chisquare(x / x.sum(), y / y.sum())

            if p <= 0.05 / len(nouns):
                rejected += 1

        scores.append(rejected / len(nouns))

    return float(np.mean(scores))


def run(lang, seed):
    filename = SAVE / f"control_{lang}_self_debias_seed{seed}.json"

    if filename.exists():
        print("SKIPPING:", filename.name, flush=True)
        return

    template_dir = (
        ROOT / "data/templates/v2" if lang == "ur"
        else ROOT / "data/templates"
    )

    templates = load_templates(
        template_dir / f"disco_templates_{lang}.json"
    )
    names = load_names(lang)
    validate(templates, names)

    rng = random.Random(seed)
    male = list(names["male"])
    female = list(names["female"])

    rng.shuffle(male)
    rng.shuffle(female)

    half = len(male) // 2
    ma, mb = male[:half], male[half:2*half]
    fa, fb = female[:half], female[half:2*half]

    runs = {
        "MF": (
            pd.DataFrame({"male": ma, "female": fa}),
            templates
        ),
        "MM": (
            pd.DataFrame({"male": ma, "female": mb}),
            {"male": templates["male"], "female": templates["male"]}
        ),
        "FF": (
            pd.DataFrame({"male": fa, "female": fb}),
            {"male": templates["female"], "female": templates["female"]}
        )
    }

    model = SelfDebiasModel(
        model_name="xlm-roberta-base",
        lang=lang
    )

    scores = {}

    for label, (nouns, tmpl) in runs.items():
        print(f"\n{lang.upper()} | seed {seed} | {label}", flush=True)
        scores[label] = disco_score(model, nouns, tmpl)
        print(f"{label}: {scores[label]:.4f}", flush=True)

    same = (scores["MM"] + scores["FF"]) / 2

    result = {
        "language": lang,
        "model": "xlm-roberta-base",
        "method": "self_debias",
        "seed": seed,
        "name_pairs_per_run": half,
        "scores": scores,
        "same_gender_average": same,
        "gender_effect": scores["MF"] - same,
        "templates_file": str(template_dir),
        "date": datetime.now(timezone.utc).isoformat()
    }

    filename.write_text(
        json.dumps(result, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    print("SAVED TO DRIVE:", filename.name, flush=True)


if __name__ == "__main__":
    for language in ["en", "ur"]:
        for seed in range(5):
            run(language, seed)
''', encoding="utf-8")

print("Script created:", script.exists())

Script created: True


In [ ]:
import subprocess
import sys

ROOT = "/content/nlp-gender-bias-urdu-main"

process = subprocess.Popen(
    [sys.executable, "-u", "self_debias_controls.py"],
    cwd=ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end="", flush=True)

code = process.wait()
print("\nExit code:", code)


Loading weights: 100%|██████████| 202/202 [00:00<00:00, 6904.32it/s]
[transformers] XLMRobertaForMaskedLM LOAD REPORT from: xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.

EN | seed 0 | MF

100%|██████████| 14/14 [00:36<00:00,  2.59s/it]
MF: 0.7500

EN | seed 0 | MM

100%|██████████| 14/14 [00:34<00:00,  2.48s/it]
MM: 0.7764

EN | seed 0 | FF

100%|██████████| 14/14 [00:35<00:00,  2.54s/it]
FF: 0.7516
SAVED TO DRIVE: control_en_self_debias_seed0.json

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 5324.79it/s]
[transformers] XLMRobertaForMaskedLM LOAD REPORT from: xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight |

In [ ]:
from pathlib import Path
import json

ROOT = Path("/content/nlp-gender-bias-urdu-main")

print("=== MODEL CHECKPOINTS ===")
for base in [Path("/content"), ROOT, Path("/content/drive/MyDrive")]:
    if base.exists():
        for p in sorted(base.glob("**/config.json")):
            if any(x in str(p).lower() for x in ["cda_en", "cda_ur"]):
                print("MODEL:", p.parent)

print("\n=== EXISTING CONTROL RESULTS ===")
for base in [ROOT / "results", Path("/content/drive/MyDrive")]:
    if base.exists():
        for p in sorted(base.glob("**/control_*.json")):
            try:
                data = json.loads(p.read_text())
                print(
                    p.name,
                    "| model:", data.get("model"),
                    "| lang:", data.get("language"),
                    "| seed:", data.get("seed")
                )
            except Exception:
                print("Could not read:", p)

=== MODEL CHECKPOINTS ===

=== EXISTING CONTROL RESULTS ===
control_en_cda_en_full.json | model: /content/cda_en_full | lang: en | seed: 4
control_en_cda_en_ur_full.json | model: /content/cda_en_ur_full | lang: en | seed: 4
control_en_cda_ur_full.json | model: /content/cda_ur_full | lang: en | seed: 4
control_en_xlm-roberta-base.json | model: xlm-roberta-base | lang: en | seed: 0
control_hi_xlm-roberta-base.json | model: xlm-roberta-base | lang: hi | seed: 0
control_ur_cda_en_full.json | model: /content/cda_en_full | lang: ur | seed: 4
control_ur_cda_en_ur_full.json | model: /content/cda_en_ur_full | lang: ur | seed: 4
control_ur_cda_hi_full.json | model: /content/cda_hi_full | lang: ur | seed: 3
control_ur_cda_hi_ur_full.json | model: /content/cda_hi_ur_full | lang: ur | seed: 3
control_ur_cda_ur_full.json | model: /content/cda_ur_full | lang: ur | seed: 4
control_ur_xlm-roberta-base.json | model: xlm-roberta-base | lang: ur | seed: 0
control_en_cda_en_full.json | model: /content/cda_

In [ ]:
from pathlib import Path

drive = Path("/content/drive/MyDrive")

for p in drive.rglob("*"):
    if p.is_file() and (
        "cda_" in p.name.lower()
        or "checkpoint" in p.name.lower()
    ):
        print(p.relative_to(drive), "|", round(p.stat().st_size / 1e6, 1), "MB")

CDA_Models/cda_hi_full.zip | 962.9 MB
CDA_Models/cda_hi_ur_full.zip | 970.9 MB
stream2_hindi_controls/control_ur_cda_hi_full_seed0.json | 0.0 MB
stream2_hindi_controls/control_ur_cda_hi_full_seed1.json | 0.0 MB
stream2_hindi_controls/control_ur_cda_hi_full_seed2.json | 0.0 MB
stream2_hindi_controls/control_ur_cda_hi_full_seed3.json | 0.0 MB
stream2_hindi_controls/control_ur_cda_hi_ur_full_seed0.json | 0.0 MB
stream2_hindi_controls/control_ur_cda_hi_ur_full_seed1.json | 0.0 MB
stream2_hindi_controls/control_ur_cda_hi_ur_full_seed2.json | 0.0 MB
stream2_hindi_controls/control_ur_cda_hi_ur_full_seed3.json | 0.0 MB


In [ ]:
from pathlib import Path

for base in [Path("/content"), Path("/content/drive/MyDrive")]:
    print(f"\n=== {base} ===")

    for p in base.rglob("*"):
        if p.is_file() and (
            p.suffix.lower() in [".zip", ".safetensors", ".bin", ".pt"]
        ):
            if p.stat().st_size > 100_000_000:
                print(
                    str(p),
                    "|",
                    round(p.stat().st_size / 1e6, 1),
                    "MB"
                )


=== /content ===
/content/cda_hi_ur_full/model.safetensors | 1113.2 MB
/content/cda_hi_full/model.safetensors | 1113.2 MB
/content/drive/MyDrive/CDA_Models/cda_hi_full.zip | 962.9 MB
/content/drive/MyDrive/CDA_Models/cda_hi_ur_full.zip | 970.9 MB
/content/drive/MyDrive/song/part 2/woman-walking-and-tossing-leaves-in-autumn-forest-2025-12-17-16-57-50-utc.zip | 119.5 MB
/content/drive/MyDrive/song/part 2/woman-on-coastal-cliffside-filmed-by-drone-2025-12-17-05-43-21-utc.zip | 243.4 MB

=== /content/drive/MyDrive ===
/content/drive/MyDrive/CDA_Models/cda_hi_full.zip | 962.9 MB
/content/drive/MyDrive/CDA_Models/cda_hi_ur_full.zip | 970.9 MB
/content/drive/MyDrive/song/part 2/woman-walking-and-tossing-leaves-in-autumn-forest-2025-12-17-16-57-50-utc.zip | 119.5 MB
/content/drive/MyDrive/song/part 2/woman-on-coastal-cliffside-filmed-by-drone-2025-12-17-05-43-21-utc.zip | 243.4 MB


In [ ]:
from pathlib import Path

ROOT = Path("/content/nlp-gender-bias-urdu-main")

print("=== TRAINING FILES ===")
for p in ROOT.rglob("*"):
    if p.is_file() and p.suffix in [".py", ".ipynb"]:
        if "train" in p.name.lower() or "cda" in str(p).lower():
            print(p.relative_to(ROOT))

print("\n=== CDA DATA ===")
for p in (ROOT / "data/cda").glob("*"):
    print(p.name, "|", round(p.stat().st_size / 1024, 1), "KB")

print("\n=== GPU ===")
import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

=== TRAINING FILES ===
notebooks/stream2_cda_training_recovered.ipynb
src/cda/__init__.py
src/cda/collect_sentences.py
src/cda/check_sheet.py
src/cda/swap.py
src/cda/translate.py

=== CDA DATA ===
pairs_hi.jsonl | 9843.9 KB
check_hi.csv | 101.0 KB
gender_pairs.tsv | 1.3 KB
check_ur.csv | 80.1 KB
en_sentences.txt | 1324.7 KB
en_pairs.jsonl | 3116.1 KB
pairs_ur.jsonl | 7644.2 KB

=== GPU ===
Tesla T4


In [ ]:
import json
from pathlib import Path

p = Path("/content/nlp-gender-bias-urdu-main/notebooks/stream2_cda_training_recovered.ipynb")
nb = json.loads(p.read_text())

for i, cell in enumerate(nb["cells"]):
    if cell["cell_type"] == "code":
        print(f"\n=== CELL {i} ===")
        print("".join(cell["source"])[:5000])


=== CELL 1 ===
from pathlib import Path
import zipfile, os
zip_path = Path('/content/nlp-gender-bias-urdu-main.zip')
if not Path('/content/nlp-gender-bias-urdu-main').exists():
    assert zip_path.exists(), 'Upload nlp-gender-bias-urdu-main.zip to /content first'
    with zipfile.ZipFile(zip_path) as z:
        z.extractall('/content')
os.chdir('/content/nlp-gender-bias-urdu-main')
print('Project:', os.getcwd())


=== CELL 3 ===

import json
import torch
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

MODEL = "xlm-roberta-base"
OUTPUT = "/content/cda_en_full"

# Load all English CDA pairs
sentences = []

with open("data/cda/en_pairs.jsonl", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        sentences.extend([row["en"], row["en_cf"]])

print("Training sentences:", len(sentences))

tokenizer = AutoTokeniz

In [ ]:
import os, sys, json, re, random, gc, shutil, subprocess
from pathlib import Path

ROOT = Path("/content/nlp-gender-bias-urdu-main")
DRIVE = Path("/content/drive/MyDrive/CDA_Models")

assert ROOT.exists(), "Project folder missing!"
assert Path("/content/drive/MyDrive").exists(), "Mount Google Drive first!"

DRIVE.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

script = r'''
import os, json, re, random, gc, sys
from pathlib import Path

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)

BASE = "xlm-roberta-base"
SEED = 42

assert torch.cuda.is_available(), "Enable T4 GPU!"

random.seed(SEED)
torch.manual_seed(SEED)

setup = sys.argv[1]
output = Path("/content") / setup

# Load English pairs
with open("data/cda/en_pairs.jsonl", encoding="utf-8") as f:
    en_rows = [json.loads(x) for x in f if x.strip()]

if setup == "cda_en_full":
    sentences = []
    for row in en_rows:
        sentences.extend([row["en"], row["en_cf"]])

    metadata = {
        "method": "CDA-EN",
        "english_pairs": len(en_rows),
        "training_sentences": len(sentences)
    }

elif setup == "cda_en_ur_full":

    with open("data/cda/pairs_ur.jsonl", encoding="utf-8") as f:
        ur_rows = [json.loads(x) for x in f if x.strip()]

    def mostly_latin(text):
        latin = len(re.findall(r"[A-Za-z]", text))
        arabic = len(re.findall(r"[\u0600-\u06FF]", text))
        return latin > arabic

    ur_valid = [
        r for r in ur_rows
        if r["ur"].strip()
        and r["ur_cf"].strip()
        and not mostly_latin(r["ur"])
        and not mostly_latin(r["ur_cf"])
        and r["ur"].strip() != r["ur_cf"].strip()
    ]

    n = min(len(en_rows), len(ur_valid))
    rng = random.Random(SEED)

    en_sample = rng.sample(en_rows, n)
    ur_sample = rng.sample(ur_valid, n)

    sentences = []

    for row in en_sample:
        sentences.extend([row["en"], row["en_cf"]])

    for row in ur_sample:
        sentences.extend([row["ur"], row["ur_cf"]])

    rng.shuffle(sentences)

    metadata = {
        "method": "CDA-EN+UR",
        "english_pairs": n,
        "urdu_pairs": n,
        "training_sentences": len(sentences)
    }

else:
    raise ValueError("Unknown setup: " + setup)

print("SETUP:", setup, flush=True)
print("TRAINING SENTENCES:", len(sentences), flush=True)

tokenizer = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForMaskedLM.from_pretrained(BASE)

dataset = Dataset.from_dict({"text": sentences})

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128,
        return_special_tokens_mask=True
    )

dataset = dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"]
)

collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

args = TrainingArguments(
    output_dir=str(output),
    max_steps=500,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    logging_steps=25,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    report_to="none",
    fp16=True,
    seed=SEED
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset,
    data_collator=collator,
    processing_class=tokenizer
)

trainer.train()

output.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(output))
tokenizer.save_pretrained(str(output))

metadata.update({
    "base_model": BASE,
    "seed": SEED,
    "max_steps": 500,
    "learning_rate": 2e-5,
    "batch_size": 4,
    "gradient_accumulation_steps": 2,
    "max_length": 128,
    "mlm_probability": 0.15,
    "fp16": True,
    "device": torch.cuda.get_device_name(0)
})

(output / "experiment.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8"
)

print("TRAINING COMPLETE:", setup, flush=True)
'''

script_path = ROOT / "src/cda/train_remaining.py"
script_path.write_text(script, encoding="utf-8")

for setup in ["cda_en_full", "cda_en_ur_full"]:

    print("\n" + "=" * 55, flush=True)
    print("STARTING:", setup, flush=True)
    print("=" * 55, flush=True)

    output = Path("/content") / setup
    backup = DRIVE / f"{setup}.zip"

    # Skip only if an existing backup is a valid ZIP
    if backup.exists() and backup.stat().st_size > 100_000_000:
        import zipfile
        with zipfile.ZipFile(backup) as z:
            if z.testzip() is None and any(
                name.endswith("model.safetensors")
                for name in z.namelist()
            ):
                print("VALID BACKUP EXISTS. SKIPPING:", setup)
                continue

    result = subprocess.run(
        [sys.executable, "-u", str(script_path), setup],
        cwd=str(ROOT)
    )

    if result.returncode != 0:
        raise RuntimeError(f"Training failed: {setup}")

    assert (output / "model.safetensors").exists(), \
        f"Model weights missing: {setup}"

    print("Creating Google Drive backup...", flush=True)

    archive = shutil.make_archive(
        str(DRIVE / setup),
        "zip",
        root_dir="/content",
        base_dir=setup
    )

    print("BACKUP SAVED:", archive, flush=True)
    print(
        "BACKUP SIZE:",
        round(Path(archive).stat().st_size / 1e6, 1),
        "MB",
        flush=True
    )

print("\nALL REMAINING CDA TRAINING COMPLETED!")


STARTING: cda_en_full
Creating Google Drive backup...
BACKUP SAVED: /content/drive/MyDrive/CDA_Models/cda_en_full.zip
BACKUP SIZE: 6913.3 MB

STARTING: cda_en_ur_full
Creating Google Drive backup...
BACKUP SAVED: /content/drive/MyDrive/CDA_Models/cda_en_ur_full.zip
BACKUP SIZE: 6934.2 MB

ALL REMAINING CDA TRAINING COMPLETED!


In [ ]:
from pathlib import Path

for name in ["cda_en_full", "cda_en_ur_full"]:
    folder = Path("/content") / name

    print(f"\n=== {name} ===")
    print("Model weights:", (folder / "model.safetensors").exists())
    print("Config:", (folder / "config.json").exists())
    print("Tokenizer:", (folder / "tokenizer_config.json").exists())
    print("Backup:", (Path("/content/drive/MyDrive/CDA_Models") / f"{name}.zip").exists())


=== cda_en_full ===
Model weights: True
Config: True
Tokenizer: True
Backup: True

=== cda_en_ur_full ===
Model weights: True
Config: True
Tokenizer: True
Backup: True


In [ ]:
import os, sys, json, shutil, subprocess, time
from pathlib import Path

ROOT = Path("/content/nlp-gender-bias-urdu-main")
DRIVE = Path("/content/drive/MyDrive/stream2_final_evaluation")
DRIVE.mkdir(parents=True, exist_ok=True)

os.chdir(ROOT)

models = [
    "cda_en_full",
    "cda_ur_full",
    "cda_en_ur_full"
]

languages = ["en", "ur"]
seeds = range(5)

def execute(cmd):
    print("\nRUNNING:", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True, cwd=str(ROOT))

# Restore Urdu checkpoint if needed
urdu_model = Path("/content/cda_ur_full")

if not (urdu_model / "model.safetensors").exists():
    backup = Path("/content/drive/MyDrive/CDA_Models/cda_ur_full.zip")
    assert backup.exists(), "Urdu model backup missing"
    shutil.unpack_archive(str(backup), "/content")

assert (urdu_model / "model.safetensors").exists()

# 1. Main DisCo evaluations
for model in models:
    model_path = f"/content/{model}"

    for lang in languages:
        tag = f"final_{model}_{lang}"

        cmd = [
            sys.executable, "-m", "src.disco.run",
            "--lang", lang,
            "--model", model_path,
            "--tokenizer", "xlm-roberta-base",
            "--tag", tag
        ]

        if lang == "ur":
            cmd += ["--templates_dir", "data/templates/v2"]

        execute(cmd)

        # Copy newly generated evaluation files
        for file in (ROOT / "results").glob(f"*{tag}*.json"):
            shutil.copy2(file, DRIVE / file.name)

# 2. Matched controls
for model in models:
    model_path = f"/content/{model}"

    for lang in languages:
        for seed in seeds:

            cmd = [
                sys.executable, "-m", "src.disco.control",
                "--lang", lang,
                "--model", model_path,
                "--seed", str(seed)
            ]

            if lang == "ur":
                cmd += ["--templates_dir", "data/templates/v2"]

            execute(cmd)

            source = (
                ROOT / "results" /
                f"control_{lang}_{model}.json"
            )

            assert source.exists(), f"Missing: {source}"

            destination = (
                DRIVE /
                f"control_{lang}_{model}_seed{seed}.json"
            )

            shutil.copy2(source, destination)

            print("SAVED:", destination.name, flush=True)

print("\nALL FINAL CDA EVALUATIONS COMPLETED!")
print("Results saved to:", DRIVE)


RUNNING: /usr/bin/python3 -m src.disco.run --lang en --model /content/cda_en_full --tokenizer xlm-roberta-base --tag final_cda_en_full_en

RUNNING: /usr/bin/python3 -m src.disco.run --lang ur --model /content/cda_en_full --tokenizer xlm-roberta-base --tag final_cda_en_full_ur --templates_dir data/templates/v2

RUNNING: /usr/bin/python3 -m src.disco.run --lang en --model /content/cda_ur_full --tokenizer xlm-roberta-base --tag final_cda_ur_full_en

RUNNING: /usr/bin/python3 -m src.disco.run --lang ur --model /content/cda_ur_full --tokenizer xlm-roberta-base --tag final_cda_ur_full_ur --templates_dir data/templates/v2

RUNNING: /usr/bin/python3 -m src.disco.run --lang en --model /content/cda_en_ur_full --tokenizer xlm-roberta-base --tag final_cda_en_ur_full_en

RUNNING: /usr/bin/python3 -m src.disco.run --lang ur --model /content/cda_en_ur_full --tokenizer xlm-roberta-base --tag final_cda_en_ur_full_ur --templates_dir data/templates/v2

RUNNING: /usr/bin/python3 -m src.disco.control --la

In [ ]:
from pathlib import Path

folder = Path("/content/drive/MyDrive/stream2_final_evaluation")

print("Folder accessible:", folder.exists())

if folder.exists():
    files = list(folder.glob("*.json"))
    print("JSON files found:", len(files))
    for f in files[:5]:
        print(f.name)